In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
import tensorflow as tf
from keras.models import Sequential, load_model
from keras.layers import GRU, Dense, Dropout, LayerNormalization, Bidirectional, LSTM, SimpleRNN
from keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau
import keras_tuner as kt
from keras.optimizers import Adam
from datetime import datetime



In [ ]:
df = pd.read_csv('enriched_timeline.csv')
X_padded = np.load("X_padded.npy", allow_pickle=True)
y_padded = np.load("y_padded.npy", allow_pickle=True)
df.drop(columns=['last_act_date'], inplace=True)

In [ ]:
df = pd.read_csv('enriched_timeline.csv')
timeline = pd.read_csv("enriched_timeline.csv", parse_dates=["Day"])
X_padded = np.load("X_padded.npy", allow_pickle=True)
y_padded = np.load("y_padded.npy", allow_pickle=True)
df.drop(columns=['last_act_date'], inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
scaler = RobustScaler()
scaled_data = scaler.fit_transform(df[["y_next24h"]])

Complete Dataset Sampling

In [ ]:
from keras.preprocessing.sequence import pad_sequences
import numpy as np

X_padded = np.load("X_padded.npy", allow_pickle=True)
y_padded = np.load("y_padded.npy", allow_pickle=True)

# Convert padded arrays back into lists (drop all-zero padded timesteps)
patient_sequences = [X[~np.all(X == 0, axis=1)] for X in X_padded]
patient_labels = [y[~np.all(X == 0, axis=1)] for X, y in zip(X_padded, y_padded)]

def resample_sequences(patient_sequences, patient_labels, 
                       max_len=None, downsample=None):
    """
    Resample patient time-series sequences with truncation and/or downsampling.

    Args:
        patient_sequences (list of np.arrays): [seq_len, features] per patient
        patient_labels (list of np.arrays): [seq_len] per patient
        max_len (int or None): keep only the last N timesteps (truncate)
        downsample (int or None): aggregate into bins of N days (e.g. 7=weekly)

    Returns:
        X_padded, y_padded (arrays ready for RNN input)
    """
    X_out, y_out = [], []

    for Xi, yi in zip(patient_sequences, patient_labels):
        # ---- 1) truncate to last N days ----
        if max_len and Xi.shape[0] > max_len:
            Xi = Xi[-max_len:]
            yi = yi[-max_len:]

        # ---- 2) downsample into bins ----
        if downsample and Xi.shape[0] > downsample:
            n_bins = Xi.shape[0] // downsample
            Xi = Xi[:n_bins * downsample].reshape(n_bins, downsample, -1).mean(axis=1)
            yi = yi[:n_bins * downsample].reshape(n_bins, downsample).max(axis=1)
            # NOTE: using mean for features, max for labels (aggression in bin)

        X_out.append(Xi)
        y_out.append(yi)

    # ---- 3) Pad sequences for RNN ----
    X_padded = pad_sequences(X_out, padding="post", dtype="float32")
    y_padded = pad_sequences(y_out, padding="post", dtype="float32")

    return X_padded, y_padded

# -------------------------------
# EXAMPLES:
# -------------------------------

# A) Last 365 days, daily
# X_padded, y_padded = resample_sequences(patient_sequences, patient_labels, max_len=365)

# B) Last 730 days, weekly bins
X_padded, y_padded = resample_sequences(patient_sequences, patient_labels, max_len=730, downsample=7)

# C) Full sequence, 3-day bins
# X_padded, y_padded = resample_sequences(patient_sequences, patient_labels, downsample=3)

print("X_padded:", X_padded.shape, "| y_padded:", y_padded.shape)


Global Balancing

In [ ]:
def balance_by_day(df, target_col="y_next24h", random_state=42):
    balanced_list = []
    for pid, g in df.groupby("Patient_ID"):
        pos_days = g[g[target_col] == 1]
        neg_days = g[g[target_col] == 0]

        n_pos = len(pos_days)
        n_neg = len(neg_days)

        if n_pos == 0:
            # No aggression for this patient -> keep a small sample of negatives
            neg_sample = neg_days.sample(min(len(neg_days), 10), random_state=random_state)
            balanced = neg_sample
        else:
            # Sample equal number of non-aggressive days
            neg_sample = neg_days.sample(n=min(n_pos, n_neg), random_state=random_state)
            balanced = pd.concat([pos_days, neg_sample])

        balanced_list.append(balanced)

    return pd.concat(balanced_list).sort_values(["Patient_ID","Day"]).reset_index(drop=True)

In [ ]:
# ---------------------------
# Balance the entire dataset BEFORE splitting
# ---------------------------
timeline_bal = balance_by_day(timeline, target_col="y_next24h")

print("Original distribution:\n", timeline["y_next24h"].value_counts())
print("Balanced distribution (all patients):\n", timeline_bal["y_next24h"].value_counts())

# ---------------------------
# Train/val/test split (by patient IDs)
# ---------------------------
patient_ids = timeline_bal["Patient_ID"].unique()
np.random.seed(42)
np.random.shuffle(patient_ids)

train_ids, val_ids, test_ids = np.split(
    patient_ids,
    [int(.7*len(patient_ids)), int(.85*len(patient_ids))]
)

train_timeline = timeline_bal[timeline_bal["Patient_ID"].isin(train_ids)]
val_timeline   = timeline_bal[timeline_bal["Patient_ID"].isin(val_ids)]
test_timeline  = timeline_bal[timeline_bal["Patient_ID"].isin(test_ids)]

print("\nBalanced distribution per split:")
print(" Train:", train_timeline["y_next24h"].value_counts().to_dict())
print(" Val:  ", val_timeline["y_next24h"].value_counts().to_dict())
print(" Test: ", test_timeline["y_next24h"].value_counts().to_dict())


Strict Balancing

In [ ]:
def enforce_split_balance(df, target_col="y_next24h", random_state=42):
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    n = min(len(pos), len(neg))
    pos_sample = pos.sample(n=n, random_state=random_state)
    neg_sample = neg.sample(n=n, random_state=random_state)
    return pd.concat([pos_sample, neg_sample]).sample(frac=1, random_state=random_state).reset_index(drop=True)

# Apply to each split
train_timeline_bal = enforce_split_balance(train_timeline)
val_timeline_bal   = enforce_split_balance(val_timeline)
test_timeline_bal  = enforce_split_balance(test_timeline)

print("Balanced Train:", train_timeline_bal["y_next24h"].value_counts().to_dict())
print("Balanced Val:  ", val_timeline_bal["y_next24h"].value_counts().to_dict())
print("Balanced Test: ", test_timeline_bal["y_next24h"].value_counts().to_dict())


X_Train & Y_Train

In [ ]:
from sklearn.model_selection import train_test_split


patient_ids = np.array([pid for pid in df['Patient_ID'].unique()])

train_ids, test_ids = train_test_split(patient_ids, test_size=0.15, random_state=42)
train_ids, val_ids  = train_test_split(train_ids, test_size=0.176, random_state=42) 
# 0.176 ≈ 15% total, so 70/15/15

pid_order = list(df['Patient_ID'].unique())
pid_to_idx = {pid:i for i,pid in enumerate(pid_order)}

train_idx = [pid_to_idx[pid] for pid in train_ids]
val_idx   = [pid_to_idx[pid] for pid in val_ids]
test_idx  = [pid_to_idx[pid] for pid in test_ids]

X_train, y_train = X_padded[train_idx], y_padded[train_idx]
X_val, y_val     = X_padded[val_idx], y_padded[val_idx]
X_test, y_test   = X_padded[test_idx], y_padded[test_idx]


Balance Sampling [50/50 appearance of patients with incident in their admission and didnt. ]

In [ ]:
import numpy as np

balanced_X = []
balanced_y = []

# Collect sequences with aggression (positives)
pos_sequences = [(X_seq, y_seq) for X_seq, y_seq in zip(X_train, y_train) if np.any(y_seq == 1)]
neg_sequences = [(X_seq, y_seq) for X_seq, y_seq in zip(X_train, y_train) if np.all(y_seq == 0)]

print(f"Original training set: {len(X_train)} sequences")
print(f" → Positive sequences: {len(pos_sequences)}")
print(f" → Negative sequences: {len(neg_sequences)}")

# Keep all positives
for X_seq, y_seq in pos_sequences:
    balanced_X.append(X_seq)
    balanced_y.append(y_seq)

# Match number of negatives (sample with replacement if fewer than positives)
pos_count = len(pos_sequences)
if len(neg_sequences) < pos_count:
    print("⚠️ Fewer negative sequences than positives. Sampling negatives with replacement.")
    sampled_neg_idx = np.random.choice(len(neg_sequences), size=pos_count, replace=True)
else:
    sampled_neg_idx = np.random.choice(len(neg_sequences), size=pos_count, replace=False)

for idx in sampled_neg_idx:
    balanced_X.append(neg_sequences[idx][0])
    balanced_y.append(neg_sequences[idx][1])

# Convert to arrays
X_train = np.array(balanced_X, dtype="float32")
y_train = np.array(balanced_y, dtype="float32")

# Diagnostics after balancing
pos_after = sum(np.any(seq == 1) for seq in y_train)
neg_after = sum(np.all(seq == 0) for seq in y_train)

print("\nBalanced training set:")
print(f" → Total sequences: {len(y_train)}")
print(f" → Positive sequences: {pos_after}")
print(f" → Negative sequences: {neg_after}")
print(f" → Class ratio: {pos_after/(pos_after+neg_after):.2f} positive / {neg_after/(pos_after+neg_after):.2f} negative")


Model Training

In [ ]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Masking, LSTM, Dense, TimeDistributed
from sklearn.utils.class_weight import compute_class_weight

# Shapes
n_timesteps = X_train.shape[1]   # max sequence length after padding
n_features = X_train.shape[2]    # number of features per timestep


model = Sequential([
    Masking(mask_value=0., input_shape=(n_timesteps, n_features)),
    Bidirectional(GRU(128, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(GRU(64, return_sequences=True)),
    Dense(1, activation="sigmoid")
])


# Compile
model.compile(
    optimizer = Adam(learning_rate=1e-4, clipnorm=1.0),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="AUC"), tf.keras.metrics.Recall(name="recall")]
)
# model.compile(optimizer='adam', loss='mean_squared_error', metrics=[r2_score])
# model.summary()

early_stop = EarlyStopping(patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8)
y_train = y_train[..., np.newaxis]
y_val   = y_val[..., np.newaxis]
y_test  = y_test[..., np.newaxis]

# Flatten y_train to 1D (ignore padding NaNs with mask)
y_train_flat = y_train.flatten()
mask = ~np.isnan(y_train_flat)
y_train_clean = y_train_flat[mask]

# # Compute class weights
# classes = np.unique(y_train_clean)
# class_weights = compute_class_weight(class_weight="balanced", 
#                                      classes=classes, 
#                                      y=y_train_clean.astype(int))
# cw_dict = {i: w for i, w in zip(classes, class_weights)}
# print("Class weights:", cw_dict)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=32,
    epochs=40,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_AUC", mode="max", patience=3, restore_best_weights=True
        )
    ]
)

print("Test evaluation:")
print(model.evaluate(X_test, y_test))


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = f"models/GRU_{timestamp}.keras"
model.save(model_filename)

In [ ]:
y_pred_probs = model.predict(X_test)
y_true = y_test.flatten()
y_pred = y_pred_probs.flatten() 


In [ ]:
# # Flatten
# y_true = y_test.flatten()
# y_pred = y_pred_probs.flatten()

# # Keep only entries where prediction is not NaN
# mask = ~np.isnan(y_pred)
# y_true = y_true[mask]
# y_pred = y_pred[mask]

# print("Kept", mask.sum(), "valid predictions out of", len(mask))


In [ ]:
# print("NaNs in y_true:", np.isnan(y_true).sum())
# print("NaNs in y_pred:", np.isnan(y_pred).sum())
# print("Shapes:", y_true.shape, y_pred.shape)


In [ ]:
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
)
import numpy as np
from sklearn.metrics import brier_score_loss

# Predictions already flattened and cleaned into y_true, y_pred

# -------------------
# Regression-style
# -------------------
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R² Score: {r2:.6f}")

# -------------------
# Classification-style
# -------------------
threshold = 0.5
y_pred_class = (y_pred >= threshold).astype(int)

auc = roc_auc_score(y_true, y_pred)
precision = precision_score(y_true, y_pred_class, zero_division=0)
recall = recall_score(y_true, y_pred_class, zero_division=0)
f1 = f1_score(y_true, y_pred_class, zero_division=0)
accuracy = accuracy_score(y_true, y_pred_class)
brier = brier_score_loss(y_true, y_pred)

print(f"Brier Score: {brier:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


<p align="center">
  <span style="font-size:3em; font-weight:bold;">Balanced Sample by Day (instead of by patient) w/ Sliding Windows</span>
</p>

Class Weighted

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Masking, Dense, Dropout, Bidirectional, GRU
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from datetime import datetime



def build_windows(df, feature_cols, target_col="y_next24h", window=180):
    X_out, y_out = [], []
    for pid, g in df.groupby("Patient_ID"):
        g = g.sort_values("Day")
        X_seq = g[feature_cols].to_numpy(dtype="float32")
        y_seq = g[target_col].to_numpy(dtype="float32")
        for i in range(window, len(y_seq)):
            X_out.append(X_seq[i-window:i])
            y_out.append(y_seq[i])
    return np.array(X_out, dtype="float32"), np.array(y_out, dtype="float32")

# Exclude ID, Day, and target
feature_cols = [
    c for c in timeline_bal.columns
    if c not in ["Patient_ID","Day","y_next24h"] and pd.api.types.is_numeric_dtype(timeline_bal[c])
]

print("Features used for training:", feature_cols)

X_train, y_train = build_windows(train_timeline, feature_cols)
X_val,   y_val   = build_windows(val_timeline, feature_cols)
X_test,  y_test  = build_windows(test_timeline, feature_cols)

print("Shapes:")
print(" Train:", X_train.shape, y_train.shape)
print(" Val:", X_val.shape, y_val.shape)
print(" Test:", X_test.shape, y_test.shape)

# ---------------------------
# Wrap into tf.data
# ---------------------------
BATCH_SIZE = 256
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(10000).batch(BATCH_SIZE)
val_ds_full = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE)
val_ds_small = val_ds_full.take(2000 // BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE)

# ---------------------------
# Define model
# ---------------------------
n_timesteps = X_train.shape[1]
n_features = X_train.shape[2]

model = Sequential([
    Bidirectional(SimpleRNN(128, return_sequences=True, recurrent_dropout=0.1, input_shape=(n_timesteps, n_features))),
    Dropout(0.2),
    Bidirectional(LSTM(128, return_sequences=True, recurrent_dropout=0.1)),
    Dropout(0.2),
    GRU(128, recurrent_dropout=0.1),
    Dropout(0.2),
    Dense(1, activation="sigmoid")  # <-- added sigmoid for probabilities
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="AUC"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.Precision(name="precision")
    ]
)

# ---------------------------
# Train
# ---------------------------
history = model.fit(
    train_ds,
    validation_data=val_ds_small,
    epochs=120,
    callbacks=[
        EarlyStopping(monitor="val_AUC", mode="max", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8)
    ]
)

# ---------------------------
# Final Evaluation
# ---------------------------
print("\nValidation evaluation (full val set):")
print(model.evaluate(val_ds_full, verbose=1))

print("\nTest evaluation (full test set):")
print(model.evaluate(test_ds, verbose=1))

# ---------------------------
# Save model
# ---------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = f"models/GRU_DayBalance_{timestamp}.keras"
model.save(model_filename)
print(f"Model saved to {model_filename}")


In [ ]:
y_pred_probs = model.predict(X_test)
y_true = y_test.flatten()
y_pred = y_pred_probs.flatten() 


In [ ]:
# y_pred_probs = model.predict(test_ds_small, verbose=1)

# # Ensure it's 1D: one probability per sample
# if y_pred_probs.ndim > 1:
#     y_pred_probs = y_pred_probs.ravel()

# # True labels from same dataset
# y_true = np.concatenate([y.numpy() for _, y in test_ds_small], axis=0)

# print("Shapes aligned:", y_true.shape, y_pred_probs.shape)



In [ ]:
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
)
import numpy as np
from sklearn.metrics import brier_score_loss

# Predictions already flattened and cleaned into y_true, y_pred

# -------------------
# Regression-style
# -------------------
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R² Score: {r2:.6f}")

# -------------------
# Classification-style
# -------------------
threshold = 0.5
y_pred_class = (y_pred >= threshold).astype(int)

auc = roc_auc_score(y_true, y_pred)
precision = precision_score(y_true, y_pred_class, zero_division=0)
recall = recall_score(y_true, y_pred_class, zero_division=0)
f1 = f1_score(y_true, y_pred_class, zero_division=0)
accuracy = accuracy_score(y_true, y_pred_class)
brier = brier_score_loss(y_true, y_pred)

print(f"Brier Score: {brier:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
)
from datetime import datetime
import numpy as np

# # Get predictions from model on full test set
# y_pred = model.predict(test_ds, verbose=1).flatten()

# # Extract the true labels from the same dataset
# y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

print("Shapes for plotting:", y_true.shape, y_pred_probs.shape)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---------------------------
# ROC Curve
# ---------------------------
fpr, tpr, _ = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("Receiver Operating Characteristic (ROC)")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.7)
plt.savefig("figures/DayBalance_ROCCurve_"+(timestamp)+".png")
plt.show()

# ---------------------------
# Precision-Recall Curve
# ---------------------------
precision, recall, _ = precision_recall_curve(y_true, y_pred)

plt.figure(figsize=(6,6))
plt.plot(recall, precision, color="purple", lw=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(True, linestyle="--", alpha=0.7)
plt.savefig("figures/DayBalance_PRCurve_"+(timestamp)+".png")
plt.show()

# ---------------------------
# Confusion Matrix (threshold = 0.5 by default)
# ---------------------------
y_pred_class = (y_pred >= 0.5).astype(int)
cm = confusion_matrix(y_true, y_pred_class)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues, values_format="d")
plt.title("Confusion Matrix (Threshold = 0.5)")
plt.savefig("figures/DayBalance_CM_"+(timestamp)+".png")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score
)
from datetime import datetime

# Ensure y_true and y_pred are aligned:
# y_pred = model.predict(test_ds).flatten()
# y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Define thresholds to test
thresholds = [0.3, 0.5, 0.7]

# For storing results
results = []

fig, axes = plt.subplots(1, len(thresholds), figsize=(15, 4))

for i, thr in enumerate(thresholds):
    # Binarise predictions at this threshold
    y_pred_class = (y_pred >= thr).astype(int)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred_class)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues, values_format="d", ax=axes[i], colorbar=False)
    axes[i].set_title(f"Threshold = {thr}")
    
    # Metrics
    precision = precision_score(y_true, y_pred_class, zero_division=0)
    recall = recall_score(y_true, y_pred_class, zero_division=0)
    f1 = f1_score(y_true, y_pred_class, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred_class)
    
    results.append({
        "Threshold": thr,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Accuracy": accuracy
    })

plt.tight_layout()
plt.savefig("figures/DayBalance_CM_Thresholds_"+timestamp+".png")
plt.show()

# Convert results to DataFrame for nice display
results_df = pd.DataFrame(results)
print("\nMetrics at different thresholds:")
print(results_df.to_string(index=False, float_format="%.4f"))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score,
    roc_curve
)

# ---------------------------
# Find best thresholds
# ---------------------------
fpr, tpr, thresholds = roc_curve(y_true, y_pred)

# Youden’s J statistic
youden = tpr - fpr
best_youden_idx = np.argmax(youden)
best_youden_thr = thresholds[best_youden_idx]

# F1-score based threshold (subsampled for speed)
grid = np.linspace(0, 1, 100)   # only check 100 thresholds
f1_scores = []
for thr in grid:
    y_pred_class = (y_pred >= thr).astype(int)
    f1_scores.append(f1_score(y_true, y_pred_class, zero_division=0))
best_f1_idx = np.argmax(f1_scores)
best_f1_thr = grid[best_f1_idx]

print(f"Best Youden threshold: {best_youden_thr:.3f}")
print(f"Best F1 threshold (approx): {best_f1_thr:.3f}")

# ---------------------------
# Compare selected thresholds
# ---------------------------
thresholds_to_plot = [0.3, 0.5, 0.7, best_youden_thr, best_f1_thr]
results = []

fig, axes = plt.subplots(1, len(thresholds_to_plot), figsize=(4*len(thresholds_to_plot), 4))

for i, thr in enumerate(thresholds_to_plot):
    y_pred_class = (y_pred >= thr).astype(int)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred_class)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues, values_format="d", ax=axes[i], colorbar=False)
    axes[i].set_title(f"Threshold = {thr:.3f}")
    
    # Metrics
    precision = precision_score(y_true, y_pred_class, zero_division=0)
    recall = recall_score(y_true, y_pred_class, zero_division=0)
    f1 = f1_score(y_true, y_pred_class, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred_class)
    
    results.append({
        "Threshold": thr,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Accuracy": accuracy
    })

plt.tight_layout()
plt.show()

# ---------------------------
# Results table
# ---------------------------
results_df = pd.DataFrame(results)
print("\nMetrics at different thresholds:")
print(results_df.to_string(index=False, float_format="%.4f"))
